# Basic Exploratory Data Analysis: MOBIE Datasets
This notebook performs structural analysis and general distribution plotting for the Portuguese EV charging station datasets.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plot style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
DATA_DIR = '../../data'
POSTOS_FILE = os.path.join(DATA_DIR, '03_interim/MOBIe_Lista_de_postos_corrected.csv')
TARIFAS_FILE = os.path.join(DATA_DIR, '01_raw/MOBIE_Tarifas.csv')

In [ ]:
# Load datasets
df_postos = pd.read_csv(POSTOS_FILE, sep=';', encoding='utf-8-sig')
df_tarifas = pd.read_csv(TARIFAS_FILE, sep=';', encoding='utf-8-sig')

print('Datasets loaded successfully.')

In [ ]:
def print_structure(df, name):
    print(f'\n--- Structure of {name} ---')
    print(f'Shape: {df.shape}')
    print('\nColumn Info:')
    print(df.info())
    print('\nFirst 5 rows:')
    display(df.head())

print_structure(df_postos, 'MOBIe_Lista_de_postos')
print_structure(df_tarifas, 'MOBIE_Tarifas')

In [ ]:
import numpy as np

SOCKET_LEVEL_COLS = [
    'POTENCIA_TOMADA', 'FORMATO_TOMADA', 'TIPO_TOMADA', 'NÍVEL DE TENSÃO',
    'TIPO_TARIFARIO', 'NIVELTENSAO', 'OPERADOR', 'MORADA', 'MUNICIPIO', 
    'TIPO_POSTO', 'UID_TOMADA', 'UID DA TOMADA', 'POTÊNCIA DA TOMADA (kW)',
    'TIPO DE TOMADA', 'FORMATO DA TOMADA', 'CIDADE', 'ESTADO DO POSTO'
]

COLS_TO_EXCLUDE = [
    'ID', 'UID_TOMADA', 'UID DA TOMADA', 'MORADA', 'MOBICHARGER', 
    'MOBICARGA', 'ULTIMA ATUALIZAÇÃO', 'Última Atualização'
]

def plot_distributions(df, df_name):
    print(f'\nProcessing distributions for {df_name}...')
    for col in df.columns:
        if any(excl.lower() in col.lower() for excl in COLS_TO_EXCLUDE):
            continue
            
        # Determine deduplication column
        dedup_col = None
        if 'UID_TOMADA' in df.columns:
            dedup_col = 'UID_TOMADA'
        elif 'UID DA TOMADA' in df.columns:
            dedup_col = 'UID DA TOMADA'
            
        # Use deduplicated dataframe for socket-level columns
        plot_df = df
        if dedup_col and col in SOCKET_LEVEL_COLS:
            plot_df = df.drop_duplicates(subset=dedup_col)
            print(f'  Deduplicating {col} by {dedup_col} ({len(plot_df)} unique entries)')
        
        # Determine if categorical or numerical
        if plot_df[col].dtype == 'object' or plot_df[col].nunique() < 20:
            # Categorical logic
            counts = plot_df[col].value_counts()
            
            # Special handling for OPERADOR: Group small counts and show all
            if 'OPERADOR' in col.upper():
                others_mask = counts < 10
                if others_mask.any():
                    others_count = counts[others_mask].sum()
                    counts = counts[~others_mask]
                    counts['Others'] = others_count
                
                order = counts.index
                plt.figure(figsize=(10, max(5, len(counts) * 0.3)))
                ax = sns.countplot(data=plot_df[plot_df[col].isin(counts.index.drop('Others', errors='ignore')) | (plot_df[col].isna() == False)], 
                                   y=col, order=order, palette='viridis')
                # Manually adjust 'Others' in the plot is tricky with countplot, simpler to use barplot for frequencies
                plt.clf()
                plt.figure(figsize=(10, max(5, len(counts) * 0.3)))
                sns.barplot(x=counts.values, y=counts.index, palette='viridis')
                plt.title(f'Distribution: {col} (Grouped <10)')
                plt.xlabel('Count / Frequency')
                
                # Add more grid lines (every 100) ONLY for OPERADOR
                max_val = counts.max()
                plt.xticks(np.arange(0, max_val + 101, 100))
            
            # Special handling for POTENCIA: Power on X axis
            elif 'POTENCIA' in col.upper() or 'POTÊNCIA' in col.upper():
                plt.figure(figsize=(12, 6))
                # Sort numerically
                try:
                    order = sorted(plot_df[col].dropna().unique(), key=lambda x: float(str(x).replace(',', '.').split()[0]))
                except:
                    order = sorted(plot_df[col].dropna().unique())
                sns.countplot(data=plot_df, x=col, order=order, palette='magma')
                plt.title(f'Distribution: {col}')
                plt.ylabel('Count / Frequency')
                plt.xticks(rotation=45)
            
            else:
                plt.figure(figsize=(10, 5))
                order = counts.index[:15]
                sns.countplot(data=plot_df, y=col, order=order, palette='viridis')
                plt.title(f'Top Categories Distribution: {col}')
                plt.xlabel('Count / Frequency')
                
        else:
            # Numerical Plot
            plt.figure(figsize=(10, 5))
            try:
                sns.histplot(plot_df[col].dropna(), kde=True, color='teal')
                plt.title(f'Numerical Distribution: {col}')
                plt.xlabel(col)
                plt.ylabel('Frequency')
            except Exception as e:
                print(f'Could not plot numerical distribution for {col}: {e}')
                plt.close()
                continue
        
        plt.tight_layout()
        plt.show()

In [ ]:
# Plot distributions for both datasets
plot_distributions(df_postos, 'MOBIe_Lista_de_postos')

In [ ]:
plot_distributions(df_tarifas, 'MOBIE_Tarifas')